# 05 · 准备考题：一万例真实病例

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → [05 准备考题 ✅] → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 起：工具、agent …
```

> **这一章的代码 10 月 7 日已经写好**（合并请求 #2）。下面的命令重复运行安全：已经下载过的只核对、不重新下载。

## 背景：没有考题，"准不准"只能靠感觉

agent 的回答是一段话加几个病名。要知道它准不准，必须有一批**已经知道正确答案**的病例，让它一例一例去做，再自动对答案。这批病例就是**评测集**（eval set）。

公司里做 agent 也是这样：**先建评测集，再开发**。招聘要求里的 "evaluation"、"eval harness"，说的就是这件事。

考题还必须**可靠**：
- **每个人拿到的完全一样**：不然你的 60% 和我的 60% 不是一回事
- **不能偷看答案**：旧项目里的诊断 agent 就栽在这里。8 月份查出来，病例的来源论文没有被挡住，系统等于看到了答案；把来源论文挡住之后，前 10 名的命中率从 86% 掉到了 40%。之前的分数好看，但是假的

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 选数据源，写死版本 | Phenopacket Store 0.1.27 | 用的是哪一份数据，永远说得清 |
| 2 | 下载并核对 | sha256 指纹 | 每个人拿到的一模一样 |
| 3 | 整理成统一格式 | pydantic 数据模型 | 每例字段一样，有问题的进不来 |
| 4 | 分成两半 | 按论文切分 | 防止边考边改、答案泄漏 |
| 5 | 看一道题 | | 知道 agent 到底要做什么 |

**花费：0。** 数据 19 MB，公开免费。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 选数据源，写死版本

### 在干什么
决定用哪一份数据当考题，并把版本写死在代码里。

### 背景
考题要满足三个条件：真实的病人、有确诊的答案、公开（别人能复现）。医院的病历满足前两条，但不能公开。

### 用了什么技术，为什么
**Phenopacket Store**：Monarch Initiative 维护的公开病例库，从已发表的论文里整理出来，每例一个文件，用的是国际标准格式（GA4GH Phenopacket）。每例有：

| 内容 | 例子 |
|---|---|
| 作者观察到的症状（标准术语 HPO） | 听力下降、卵巢功能早衰…… |
| 确诊的病（OMIM 编号） | Perrault 综合征 7 型 |
| 致病基因和变异 | DAP3 |
| 来源论文（PMID） | PMID:39701103 |

症状和诊断是作者在论文里分别记录的，不是从"某个病应该有哪些症状"的数据库里反推出来的。这一点让它可以当考题。

**局限**：都是**已经确诊**的病例（确诊了才会写成论文）。所以在这上面测出的准确率，是真实门诊里的**上限**。

**版本写死**：用 0.1.27 版（2026 年 6 月发布）。版本号和文件指纹写在数据源清单 [`evals/datasets.toml`](../../evals/datasets.toml) 里（06 号细讲）；数据放在 `data/raw/phenopacket-store/0.1.27/`，不进 git。

### 公司里什么情况怎么做
- **官方出了新版本**：改版本号和指纹，新数据进新文件夹，旧的不动。哪个结果用的是哪一版，永远查得到
- **加一个新数据源**：写清楚从哪来、哪个版本、怎么核对，和这一份一样

### 面试怎么说
> "The eval set is Phenopacket Store: about ten thousand published cases with observed phenotypes and confirmed diagnoses. The release is pinned in code with its checksum, so every result traces back to an exact dataset version."

---

## 第 2 步 · 下载并核对

### 在干什么
一条命令下载数据，下载完核对指纹，确认和官方发布的一个字节都不差。

### 背景
数据不进代码记录（太大），但要保证每个人、每台电脑拿到的一模一样。下载可能中途断掉、文件可能被换过，光看文件名看不出来。

### 用了什么技术，为什么
**指纹**（sha256）：把文件内容算成一串 64 位的字符。文件哪怕只改了一个字节，指纹就完全不一样。

`raredx data fetch` 做的事：
1. 已经有这个文件：只核对指纹，不重新下载
2. 没有：下载到一个临时文件
3. 核对指纹：对上了，才改成正式的名字；**对不上就删掉、不使用**，不会留下半个文件

> 我顺手核对过：旧项目里那份同名文件，指纹和官方的一模一样。

**重复运行结果不变、不重复下载**，这叫**幂等**（idempotent）。数据管道必须这样，因为失败重跑、定时任务都会让同一个操作执行好几次。

### 公司里什么情况怎么做
- **指纹对不上**：先别用。重新下载一次；还不对，去官方查是不是换了文件
- **官方自己提供了指纹**：再和官方的比一次

### 执行

In [ ]:
run("uv run raredx data fetch")

### 结果怎么看
`ok  data/raw/phenopacket-store/0.1.27/all_phenopackets.zip  (release 0.1.27, sha256 verified)`：指纹对上了。已经下载过的，不到一秒就返回。

### 面试怎么说
> "The fetch step is idempotent and verifies a pinned SHA-256; a mismatched download is deleted rather than used."

---

## 第 3 步 · 整理成统一格式

### 在干什么
把 10,377 个 JSON 文件，整理成同一种格式的"病例"。

### 背景
原始文件层层嵌套，写法也不完全统一。最容易出错的一点：每个症状有一个标记 `excluded`，意思是"**医生查过，病人没有**"。数据里的第一个病例，37 个症状里有 33 个是"查过但没有"。如果不看这个标记，系统会以为病人有这 33 个症状，诊断就全错了。

旧项目的加载代码这一点处理对了，这里照着做。

### 用了什么技术，为什么
**pydantic**：用来定义"一条记录长什么样"。我们定义了一个 `Case`（病例），读进来的每一例都必须符合它：字段缺了、类型不对，当场报错，而不是悄悄带着错误往下走。

| 字段 | 内容 |
|---|---|
| `hpo_ids` / `hpo_labels` | 症状（只保留**有**的，"查过但没有"的去掉） |
| `disease_id` / `disease_label` | 确诊的病（答案） |
| `gene` | 致病基因（69 例没有，比如染色体缺失） |
| `pmid` | 来源论文 |
| `split` | 分在哪一半（第 4 步） |

**有问题的跳过，并记下原因**，不猜、不补。

### 公司里什么情况怎么做
- **原始数据原样保存**，整理写成代码。整理规则改了，重新跑一遍就行
- **跳过的数据要计数**：知道丢了多少、为什么，而不是悄悄少了

### 执行
读全部原始文件，整理好存起来（`build`），再打印统计（`stats`）：

In [ ]:
run("uv run raredx data build")
run("uv run raredx data stats")

### 结果怎么看
- `build`：`ok  data/processed/.../cases.jsonl  (10374 cases, ...)`：整理好的病例存进了 `data/processed`（这一层 06 号讲）
- `cases 10374`：可以用的病例
- `skipped {'no_phenotype': 3}`：3 例被跳过，原因是一个"有"的症状都没有
- `780` 种病、`699` 个基因、来自 `1733` 篇论文
- 每例的症状数，中位数是 7（最少 1 个，最多 66 个）

### 面试怎么说
> "Raw files are parsed into a validated pydantic model; excluded phenotypes are dropped, and every skipped record is counted with a reason."

---

## 第 4 步 · 分成两半：按论文切

### 在干什么
把病例分成两半：**开发集**（平时开发时用）和**留出集**（全部做完以后，最后考一次）。

### 背景
开发时，你会不停地看结果、改提示词、调规则。如果一直拿同一批题看分数，系统会慢慢"背熟"这批题，分数越来越高，但换一批新题就不行了。所以要留一半，**开发时一眼都不看**，最后才考。

还有一个更隐蔽的问题：**同一篇论文里的几个病人，常常是一家人**，症状的写法也一样。如果一个分在开发集、一个分在留出集，等于提前看过了答案。

### 用了什么技术，为什么
**按论文切，不按病例切**：同一篇论文的病人，永远在同一边。

**由代码决定，不由人挑**：用论文编号算一个哈希值，单数进开发集、双数进留出集。分法固定不变，没有人能"把难的题挪到另一边"。

结果是 **6,014 / 4,360**，不是一半一半：有几篇大论文一篇就有几百例，整篇落在了同一边。这是可以接受的代价，比均匀但漏答案好。

### 公司里什么情况怎么做
- **留出集只在设计定稿后用一次**。看了留出集的错题再回去改，它就变成了第二个开发集，不能再当最后的考试
- **切分的单位要选对**：同一个人的多次就诊、同一家医院的病例、同一篇论文，都不能跨两边。机器学习面试里常问的"数据泄漏"（data leakage）就是这个

### 执行
看贡献病例最多的 5 篇论文，分别落在哪一边：

In [ ]:
from collections import Counter
from raredx.data import cases as case_store, phenopackets as pp, sources

source = sources.load_sources(ROOT / "evals/datasets.toml")["phenopacket-store"]
cases = case_store.read(sources.processed_dir(source, ROOT / "data"))
for pmid, n in Counter(c.pmid for c in cases).most_common(5):
    print(f"{pmid:<16} {n:>4} 例  → {'开发集' if pp.split_of(pmid) == 'design' else '留出集'}")

### 结果怎么看
最多的一篇有 462 例，前 5 篇恰好都在开发集。几百例的大论文整篇落在一边，所以两边不一样大。

### 面试怎么说
> "The design/held-out split is by source paper, assigned by a hash so nobody can choose; patients from one report never straddle it. The held-out half is used once, when the design is final."

---

## 第 5 步 · 一道题长什么样

### 在干什么
挑一个病例，看看 agent 要做的题到底是什么。

### 执行

In [ ]:
c = next(c for c in cases if c.id == "PMID_39701103_Proband_F3")
print("症状（题目）：")
for label in c.hpo_labels:
    print("  -", label)
print("\n确诊（答案）：", c.disease_label, f"（{c.disease_id}）")
print("致病基因：  ", c.gene)
print("来源论文：  ", c.pmid)
print("分在：      ", "开发集" if c.split == "design" else "留出集")

### 结果怎么看
- **题目**是 5 个症状：原发性闭经、卵巢功能早衰、双侧感音神经性听力下降、轻度智力障碍、全面发育迟缓
- **答案**是 Perrault 综合征 7 型，基因 DAP3

agent 要做的，就是**只看症状**，说出这个病和基因。有经验的医生看到"听力下降 + 卵巢功能不全"会想到 Perrault；agent 要靠查资料做到同样的事。

**还有一点很关键：答案就写在来源论文 PMID:39701103 里。** 以后 agent 去查文献时，必须把这篇论文挡住，否则就是偷看答案。这是做查文献工具时的第一条要求。

### 面试怎么说
> "Each case is a phenotype list with a confirmed diagnosis. The answer is in the source paper, so the literature tool blocks each case's own publication."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 数据源 | Phenopacket Store，版本写死 | 结果永远能追溯到哪一份数据 |
| 2 下载核对 | sha256、幂等 | 每个人的数据一模一样 |
| 3 统一格式 | pydantic | 有问题的数据进不来，跳过的有记录 |
| 4 两半 | 按论文切分 | 防止边考边改和答案泄漏 |
| 5 一道题 | | 知道 agent 要做什么 |

**一句话**：考题准备好了：一万多例真实病例，版本固定、来源可查，平时用的和最后考的分开放。

## 面试题

1. **How do you evaluate an LLM agent?**
   > "Build the eval set first: cases with known answers, a pinned version and an automatic scorer. Develop against one split, keep a held-out split for the final run, and compare against simple baselines so you know what the agent actually adds."

2. **What is data leakage, and how did you guard against it?**
   > "Information about the answer reaching the system. Here that means patients from the same paper on both sides of the split, or the agent reading the case's own source paper. The split is by paper, and the literature tool blocks each case's publication. In the earlier project, removing exactly this leak dropped top-10 accuracy from 86% to 40%."

3. **Why pin the dataset version and checksum?**
   > "Reproducibility: any number must trace back to the exact inputs. A new release goes into a new folder, and switching to it is a deliberate change."

**下一章 06 · 评测集**：调参用哪 200 道题、最后考哪 4,360 道，题号清单冻结进 git，数据分层存放。